# Hoop Pendulum Simulation

**System:** A pendulum bob of mass $m_p$ swings inside a hoop of radius $R$ and mass $m_h$ that rolls without slipping on a flat surface. The cart/body has mass $M$.

**Generalised coordinates:** $\theta_1$ (hoop rotation) and $\theta_2$ (pendulum angle).  
Cart position is recovered from the rolling constraint: $x = -R\,\theta_1$.

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

%matplotlib inline

## Parameters

In [ ]:
M   = 1.0   # Cart/body mass [kg]
m_p = 0.5   # Pendulum mass [kg]
l   = 0.3   # Pendulum length [m]
R   = 0.5   # Wheel/hoop radius [m]
m_h = 0.3   # Hub/wheel mass [kg]
g   = 9.81  # Gravity [m/sÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Â¦Ãƒâ€šÃ‚Â¡ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â²]

## Equations of Motion & ODE Solver

The reduced $2\times2$ mass matrix and right-hand side are:

$$M_{\text{red}}\,\ddot{q} = R_{\text{vec}}$$

$$M_{\text{red}} = \begin{bmatrix} (M+m_h)R^2 & m_p l R \sin\theta_2 \\ m_p l R \sin\theta_2 & m_p l^2 \end{bmatrix}, \quad R_{\text{vec}} = \begin{bmatrix} -m_p l R \dot{\theta}_2^2 \cos\theta_2 \\ -m_p g l \cos\theta_2 \end{bmatrix}$$

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp

# Initial conditions: [theta1, theta1_dot, theta2, theta2_dot]
# theta2 is measured from the +x axis (horizontal), so theta2 = pi/2 is upright.
y0 = [0.0, 0.0, 2.09, -4.0]   # small tilt above horizontal

def eom(t, y):
    """Open-loop EOM for pendulum on a rolling hoop (Lagrangian, pure rolling).

    Generalised coordinates: q = [theta1, theta2]
      theta1 : hoop rotation (rolling constraint x = -R*theta1)
      theta2 : pendulum angle from horizontal +x axis (upright = +pi/2)
    """
    th1, th1dot, th2, th2dot = y
    s2 = np.sin(th2); c2 = np.cos(th2)

    Mred = np.array([
        [(2*m_h + m_p) * R**2,   m_p * l * R * s2],
        [ m_p * l * R * s2,        m_p * l**2      ]
    ])
    Rvec = np.array([
        -m_p * l * R * c2 * th2dot**2,   # no external torque (open-loop)
        -m_p * g * l * c2
    ])
    q_ddot = np.linalg.solve(Mred, Rvec)
    return [th1dot, q_ddot[0], th2dot, q_ddot[1]]

sol = solve_ivp(eom, (0.0, 10.0), y0,
                method='RK45', rtol=1e-8, atol=1e-10,
                dense_output=True)
t = sol.t
y = sol.y.T
print(f'Solved: {len(t)} steps')


## Results: Generalized Coordinates

In [ ]:
th1  = y[:, 0]
th1d = y[:, 1]
th2  = y[:, 2]
th2d = y[:, 3]

# Recover cart position from rolling constraint: x = -R * theta1
x = -R * th1

fig, axes = plt.subplots(3, 1, figsize=(9, 6), sharex=True)

axes[0].plot(t, x,   'b')
axes[0].set_ylabel('x [m]')
axes[0].set_title('Generalised Coordinates')
axes[0].grid(True)

axes[1].plot(t, th1, 'g')
axes[1].set_ylabel(r'$\theta_1$ [rad]')
axes[1].grid(True)

axes[2].plot(t, th2, 'r')
axes[2].set_ylabel(r'$\theta_2$ [rad]')
axes[2].set_xlabel('Time [s]')
axes[2].grid(True)

plt.tight_layout()
plt.show()

## Animation

Displays an inline animation of the hoop rolling on the ground with the pendulum swinging inside.  
Set `save_mp4=True` to save an MP4 file instead (requires `ffmpeg` installed).

In [ ]:
def animate_hoop_pendulum(t, y, R, l, fps=30, save_mp4=False,
                          filename='hoop_pendulum.mp4',
                          follow=False, view_width=None,
                          th2_ref=None, max_frames=800):
    """
    Animate a pendulum on a rolling hoop.
    theta2 is measured from horizontal:
      xb = xc + l*cos(theta2),  yb = R + l*sin(theta2)

    If follow=True, the x-axis tracks the cart with a window of width view_width
    (defaults to 6*(R+l)) so the hoop stays a sensible size on screen.

    If th2_ref is provided (array same length as t), a dashed ghost pendulum
    is drawn at the reference angle sharing the same cart position.

    max_frames caps embedded notebook animations so long simulations do not
    exceed Jupyter's default animation size limit.
    """
    import numpy as np
    import matplotlib.pyplot as plt
    import matplotlib.animation as animation

    th1 = y[:, 0]
    th2 = y[:, 2]
    xc_a = -R * th1
    yc_a = np.full_like(xc_a, R)
    xb_a = xc_a + l * np.cos(th2)
    yb_a = R + l * np.sin(th2)

    show_ref = th2_ref is not None
    if show_ref:
        th2_ref = np.asarray(th2_ref)
        xb_ref_a = xc_a + l * np.cos(th2_ref)
        yb_ref_a = R + l * np.sin(th2_ref)

    t0, tf = float(t[0]), float(t[-1])
    requested_frames = max(2, int(round((tf - t0) * fps)))
    n_frames = min(requested_frames, max_frames)
    effective_fps = n_frames / max(tf - t0, 1e-12)
    t_uniform = np.linspace(t0, tf, n_frames)
    xc_u = np.interp(t_uniform, t, xc_a)
    yc_u = np.interp(t_uniform, t, yc_a)
    xb_u = np.interp(t_uniform, t, xb_a)
    yb_u = np.interp(t_uniform, t, yb_a)
    if show_ref:
        xb_ref_u = np.interp(t_uniform, t, xb_ref_a)
        yb_ref_u = np.interp(t_uniform, t, yb_ref_a)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.set_aspect('equal')
    pad = 0.5 * (R + l)
    if follow:
        if view_width is None:
            view_width = 6.0 * (R + l)
        half_w = view_width / 2.0
        ax.set_xlim(xc_u[0] - half_w, xc_u[0] + half_w)
    else:
        ax.set_xlim(min(xc_a.min(), xb_a.min()) - pad,
                    max(xc_a.max(), xb_a.max()) + pad)
    ax.set_ylim(-pad, R + l + pad)
    ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='k', lw=0.8)

    phi = np.linspace(0, 2 * np.pi, 100)
    hoop_line, = ax.plot([], [], 'b-', lw=1.5)
    rim_pt,    = ax.plot([], [], 'bo', ms=4)
    rod_line,  = ax.plot([], [], 'k-', lw=2)
    bob_pt,    = ax.plot([], [], 'ro', ms=10)
    ctr_pt,    = ax.plot([], [], 'k+', ms=8)
    time_txt   = ax.text(0.02, 0.95, '', transform=ax.transAxes)

    if show_ref:
        ref_rod,  = ax.plot([], [], '--', color='gray', lw=1.5, alpha=0.6,
                            label='reference')
        ref_bob,  = ax.plot([], [], 'o',  color='gray', ms=8,  alpha=0.6)
        ax.legend(loc='upper right', fontsize=8)
        extra_artists = (ref_rod, ref_bob)
    else:
        extra_artists = ()

    def init():
        hoop_line.set_data([], [])
        rim_pt.set_data([], [])
        rod_line.set_data([], [])
        bob_pt.set_data([], [])
        ctr_pt.set_data([], [])
        time_txt.set_text('')
        for a in extra_artists:
            a.set_data([], [])
        return (hoop_line, rim_pt, rod_line, bob_pt, ctr_pt, time_txt) + extra_artists

    def update(k):
        cx = xc_u[k]; cy = yc_u[k]
        hoop_line.set_data(cx + R * np.cos(phi), cy + R * np.sin(phi))
        th1_k = np.interp(t_uniform[k], t, th1)
        rim_pt.set_data([cx + R * np.cos(-th1_k + np.pi/2)],
                        [cy + R * np.sin(-th1_k + np.pi/2)])
        ctr_pt.set_data([cx], [cy])
        rod_line.set_data([cx, xb_u[k]], [cy, yb_u[k]])
        bob_pt.set_data([xb_u[k]], [yb_u[k]])
        time_txt.set_text(f't = {t_uniform[k]:.2f} s')
        if show_ref:
            ref_rod.set_data([cx, xb_ref_u[k]], [cy, yb_ref_u[k]])
            ref_bob.set_data([xb_ref_u[k]], [yb_ref_u[k]])
        if follow:
            ax.set_xlim(cx - half_w, cx + half_w)
            return ()
        return (hoop_line, rim_pt, rod_line, bob_pt, ctr_pt, time_txt) + extra_artists

    anim = animation.FuncAnimation(fig, update, init_func=init,
                                   frames=n_frames, interval=1000/effective_fps,
                                   blit=not follow)
    plt.close(fig)
    if save_mp4:
        anim.save(filename, fps=effective_fps, dpi=150)

    return HTML(anim.to_jshtml())


In [ ]:
animate_hoop_pendulum(t, y, R, l, fps=30, save_mp4=False)


## PID Trajectory Tracking - Revolving Pendulum Motor

This version does not command a fixed pendulum lean angle. The pendulum is free to revolve.

The outer controller computes the internal motor torque from sphere position error and pendulum state:

$$u = K_x e_x + K_{xd} e_{\dot{x}} + K_\theta \sin(	\theta_2) + K_{\dot{	\theta}}\dot{	\theta}_2$$

The generalized input is

$$Q = [-u, u]^T$$

The $-u$ term is the equal/opposite motor reaction on the sphere coordinate, not an external torque source applied to the sphere.


In [ ]:
import numpy as np
from scipy.integrate import solve_ivp

# Default parameters, used if the parameter cell above has not been run.
M = globals().get("M", 1.0)
m_p = globals().get("m_p", 0.5)
l = globals().get("l", 0.3)
R = globals().get("R", 0.5)
m_h = globals().get("m_h", 0.3)
g = globals().get("g", 9.81)

# Desired sphere/cart trajectory x_d(t)
# Choose: 'smooth_step' or 'sinusoid'
trajectory_type = 'sinusoid'

# Smooth-step settings
step_distance = 4.0     # [m]
step_duration = 10.0     # [s]

# Sinusoidal track settings
sine_amplitude = 4.0   # [m]
sine_frequency = 0.1   # [Hz]
sine_offset = 0.0       # [m]


def x_ref_fun(t):
    if trajectory_type == 'sinusoid':
        w = 2.0 * np.pi * sine_frequency
        return sine_offset + sine_amplitude * np.sin(w * t)

    tt = np.minimum(t, step_duration)
    return step_distance * (1.0 - np.cos(np.pi * tt / step_duration)) / 2.0


def xd_ref_fun(t):
    if trajectory_type == 'sinusoid':
        w = 2.0 * np.pi * sine_frequency
        return sine_amplitude * w * np.cos(w * t)

    if t <= step_duration:
        return step_distance * (np.pi / step_duration) * np.sin(np.pi * t / step_duration) / 2.0
    return 0.0


def xdd_ref_fun(t):
    if trajectory_type == 'sinusoid':
        w = 2.0 * np.pi * sine_frequency
        return -sine_amplitude * w**2 * np.sin(w * t)

    if t <= step_duration:
        return step_distance * (np.pi / step_duration)**2 * np.cos(np.pi * t / step_duration) / 2.0
    return 0.0


# Controller gains
# These gains command motor torque directly, so the pendulum is free to revolve.
Kpx = 5.0
Kix = 0.0
Kdx = 3.0
Ktheta = -0.05     # phase feedback [N.m]
Ktheta_dot = 0.01  # angular-rate feedback [N.m.s/rad]

u_max_track = 20.0      # smooth internal motor torque limit [N.m]
ix_max = 5.0            # outer integral scale [m.s]
u_pid_sign = 1.0        # flip to -1 if sphere moves opposite the reference


def smooth_limit(value, limit):
    """Smooth saturation with no corner/discontinuity in slope."""
    return limit * np.tanh(value / limit)


def reference_values(t):
    return x_ref_fun(t), xd_ref_fun(t), xdd_ref_fun(t)


def tracking_torque(t, th1, th1dot, th2, th2dot, ix):
    x = -R * th1
    xd = -R * th1dot
    x_ref, xd_ref, _ = reference_values(t)

    ex = x_ref - x
    exd = xd_ref - xd
    ix_limited = smooth_limit(ix, ix_max)

    u_track = Kpx * ex + Kix * ix_limited + Kdx * exd
    u_phase = Ktheta * np.sin(th2) + Ktheta_dot * th2dot
    u_raw = u_pid_sign * (u_track + u_phase)
    u = smooth_limit(u_raw, u_max_track)
    return u, ex, u_track, u_phase


def eom_trajectory_pid(t, state):
    # state = [theta1, theta1_dot, theta2, theta2_dot, int_x_error]
    th1, th1dot, th2, th2dot, ix = state
    s2 = np.sin(th2); c2 = np.cos(th2)

    u, ex, _, _ = tracking_torque(t, th1, th1dot, th2, th2dot, ix)

    Mred = np.array([
        [(2*m_h + m_p) * R**2,   m_p * l * R * s2],
        [ m_p * l * R * s2,        m_p * l**2      ]
    ])

    # Internal motor torque only: equal/opposite generalized torques.
    # The sphere coordinate gets -u only as the motor reaction, not as an external drive.
    Rvec = np.array([
        -u - m_p * l * R * c2 * th2dot**2,
        u - m_p * g * l * c2
    ])
    q_ddot = np.linalg.solve(Mred, Rvec)

    ix_dot = ex
    return [th1dot, q_ddot[0], th2dot, q_ddot[1], ix_dot]


# Initial condition. The pendulum is not held near upright; it is free to rotate.
# Try sensitive cases here, for example theta2=0.1, theta2dot=4.0
# or theta2=2.09, theta2dot=-4.0.
theta2_init_track = np.pi / 2
theta2dot_init_track = 4.0
y0_track = [0.0, 0.0, theta2_init_track, theta2dot_init_track, 0.0]

sol_track = solve_ivp(
    eom_trajectory_pid, (0.0, 30.0), y0_track,
    method='RK45', rtol=1e-5, atol=1e-7, max_step=0.01, dense_output=True
)

t_track = np.linspace(sol_track.t[0], sol_track.t[-1], 3001)
y_track = sol_track.sol(t_track).T
print(f'Solved: {len(sol_track.t)} adaptive steps, resampled to {len(t_track)} smooth frames | final x = {-R*y_track[-1, 0]:+.3f} m | target = {x_ref_fun(t_track[-1]):+.3f} m | pendulum revolutions = {(y_track[-1, 2] - y_track[0, 2])/(2*np.pi):+.1f}')


In [ ]:
# Plot trajectory tracking results
import numpy as np
import matplotlib.pyplot as plt

th1t  = y_track[:, 0]
th1dt = y_track[:, 1]
th2t  = y_track[:, 2]
th2dt = y_track[:, 3]

x_track = -R * th1t
xd_track = -R * th1dt
th2_unwrapped = np.unwrap(th2t)
pendulum_revs = (th2_unwrapped - th2_unwrapped[0]) / (2.0 * np.pi)

x_ref_hist = np.zeros_like(t_track)
xd_ref_hist = np.zeros_like(t_track)
u_track_hist = np.zeros_like(t_track)
u_position_hist = np.zeros_like(t_track)
u_phase_hist = np.zeros_like(t_track)

for i, ti in enumerate(t_track):
    th1, th1d, th2, th2d, ix = y_track[i]
    x_ref_hist[i], xd_ref_hist[i], _ = reference_values(ti)
    u_track_hist[i], _, u_position_hist[i], u_phase_hist[i] = tracking_torque(ti, th1, th1d, th2, th2d, ix)

err = x_track - x_ref_hist
rms_err = np.sqrt(np.mean(err[t_track > 5.0]**2)) if np.any(t_track > 5.0) else np.sqrt(np.mean(err**2))

fig, axs = plt.subplots(5, 1, figsize=(9, 11), sharex=True)

axs[0].plot(t_track, x_track, label='x')
axs[0].plot(t_track, x_ref_hist, 'r--', label='x reference')
axs[0].set_ylabel('sphere x [m]')
axs[0].legend(fontsize=8); axs[0].grid(True, alpha=0.3)

axs[1].plot(t_track, xd_track, label='xdot')
axs[1].plot(t_track, xd_ref_hist, 'r--', label='xdot reference')
axs[1].set_ylabel('velocity [m/s]')
axs[1].legend(fontsize=8); axs[1].grid(True, alpha=0.3)

axs[2].plot(t_track, pendulum_revs)
axs[2].set_ylabel(r'pendulum revs')
axs[2].grid(True, alpha=0.3)

axs[3].plot(t_track, th2dt, color='teal')
axs[3].set_ylabel(r'$\dot{\theta}_2$ [rad/s]')
axs[3].grid(True, alpha=0.3)

axs[4].plot(t_track, u_track_hist, label='total u')
axs[4].plot(t_track, u_position_hist, '--', alpha=0.7, label='x feedback')
axs[4].plot(t_track, u_phase_hist, ':', alpha=0.8, label='theta feedback')
axs[4].axhline( u_max_track, color='r', ls='--', alpha=0.5)
axs[4].axhline(-u_max_track, color='r', ls='--', alpha=0.5)
axs[4].set_ylabel('internal u [N.m]')
axs[4].legend(fontsize=8)
axs[4].set_xlabel(f't [s]   RMS error after 5 s = {rms_err:.3f} m')
axs[4].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()


In [ ]:
# Animate the trajectory-tracking result
animate_hoop_pendulum(t_track, y_track[:, :4], R, l, fps=20, save_mp4=False,
                      follow=False, max_frames=800)


## TVLQR Trajectory Tracking - Revolving Pendulum

A fixed-equilibrium LQR holds the pendulum near one angle, so it cannot give the same revolving motion as the PID controller. For a revolving pendulum, use **time-varying LQR (TVLQR)** around a nominal revolving trajectory.

Workflow:

1. Generate a nominal revolving trajectory with an internal-motor controller.
2. Linearize the nonlinear dynamics along that trajectory.
3. Solve a backward Riccati recursion to get time-varying gains.
4. Simulate the nonlinear system with $u = u_{nom} - K(t)(y-y_{nom})$.

The actuator model is still internal:

$$Q = [-u, u]^T$$


In [ ]:
import numpy as np
from scipy.integrate import solve_ivp

M = globals().get("M", 1.0)
m_p = globals().get("m_p", 0.5)
l = globals().get("l", 0.3)
R = globals().get("R", 0.5)
m_h = globals().get("m_h", 0.3)
g = globals().get("g", 9.81)

# TVLQR around a revolving nominal trajectory

tvlqr_trajectory_type = 'sinusoid'  # 'sinusoid' or 'smooth_step'

# Keep this modest; revolving internal-mass systems are very nonlinear.
tvlqr_sine_amplitude = 4.0
tvlqr_sine_frequency = 0.1
tvlqr_sine_offset = 0.0

tvlqr_step_distance = 1.0
tvlqr_step_duration = 8.0

T_tvlqr = 30.0
N_tvlqr = 1201
t_grid = np.linspace(0.0, T_tvlqr, N_tvlqr)
dt_grid = t_grid[1] - t_grid[0]


def x_ref_tvlqr(t):
    if tvlqr_trajectory_type == 'sinusoid':
        w = 2.0 * np.pi * tvlqr_sine_frequency
        return tvlqr_sine_offset + tvlqr_sine_amplitude * np.sin(w * t)
    tt = np.minimum(t, tvlqr_step_duration)
    return tvlqr_step_distance * (1.0 - np.cos(np.pi * tt / tvlqr_step_duration)) / 2.0


def xd_ref_tvlqr(t):
    if tvlqr_trajectory_type == 'sinusoid':
        w = 2.0 * np.pi * tvlqr_sine_frequency
        return tvlqr_sine_amplitude * w * np.cos(w * t)
    if t <= tvlqr_step_duration:
        return tvlqr_step_distance * (np.pi / tvlqr_step_duration) * np.sin(np.pi * t / tvlqr_step_duration) / 2.0
    return 0.0


def smooth_limit(value, limit):
    return limit * np.tanh(value / limit)


def internal_f(y4, u):
    th1, th1dot, th2, th2dot = y4
    s2 = np.sin(th2)
    c2 = np.cos(th2)
    Mred = np.array([
        [(2*m_h + m_p) * R**2,   m_p * l * R * s2],
        [ m_p * l * R * s2,        m_p * l**2      ]
    ])
    Rvec = np.array([
        -u - m_p * l * R * c2 * th2dot**2,
        u - m_p * g * l * c2
    ])
    th1ddot, th2ddot = np.linalg.solve(Mred, Rvec)
    return np.array([th1dot, th1ddot, th2dot, th2ddot])


# Nominal revolving controller used only to create the trajectory that TVLQR tracks.
Kpx_nom = 5.0
Kdx_nom = 3.0
Ktheta_nom = -0.05
Ktheta_dot_nom = 0.01
u_nom_limit = 20.0


def nominal_u(t, y4):
    th1, th1dot, th2, th2dot = y4
    x = -R * th1
    xd = -R * th1dot
    ex = x_ref_tvlqr(t) - x
    exd = xd_ref_tvlqr(t) - xd
    u_raw = Kpx_nom * ex + Kdx_nom * exd + Ktheta_nom * np.sin(th2) + Ktheta_dot_nom * th2dot
    return smooth_limit(u_raw, u_nom_limit)


def eom_nominal(t, y4):
    return internal_f(y4, nominal_u(t, y4))


# Start with spin so the nominal pendulum revolves.
y0_nom = np.array([0.0, 0.0, np.pi / 2.0, 4.0])
sol_nom = solve_ivp(
    eom_nominal, (0.0, T_tvlqr), y0_nom,
    method='RK45', rtol=1e-6, atol=1e-8, max_step=0.01, dense_output=True
)
y_nom = sol_nom.sol(t_grid).T
u_nom = np.array([nominal_u(ti, yi) for ti, yi in zip(t_grid, y_nom)])


def linearize_along(y4, u, eps_x=1e-6, eps_u=1e-6):
    A = np.zeros((4, 4))
    B = np.zeros((4, 1))
    for j in range(4):
        dy = np.zeros(4)
        dy[j] = eps_x
        A[:, j] = (internal_f(y4 + dy, u) - internal_f(y4 - dy, u)) / (2.0 * eps_x)
    B[:, 0] = (internal_f(y4, u + eps_u) - internal_f(y4, u - eps_u)) / (2.0 * eps_u)
    return A, B


A_list = np.zeros((N_tvlqr, 4, 4))
B_list = np.zeros((N_tvlqr, 4, 1))
for k in range(N_tvlqr):
    A_list[k], B_list[k] = linearize_along(y_nom[k], u_nom[k])

# State error is [x, xdot, theta2 phase, theta2dot]. Because y stores theta1,
# the equivalent theta1 weights are scaled by x = -R*theta1.
Q_tvlqr = np.diag([80.0 * R**2, 20.0 * R**2, 8.0, 0.8])
Qf_tvlqr = np.diag([150.0 * R**2, 40.0 * R**2, 15.0, 1.5])
R_tvlqr = np.array([[0.8]])

S = Qf_tvlqr.copy()
K_grid = np.zeros((N_tvlqr, 1, 4))
for k in range(N_tvlqr - 2, -1, -1):
    A = A_list[k]
    B = B_list[k]
    Ad = np.eye(4) + A * dt_grid
    Bd = B * dt_grid
    Qd = Q_tvlqr * dt_grid
    Rd = R_tvlqr * dt_grid
    G = Rd + Bd.T @ S @ Bd
    K = np.linalg.solve(G, Bd.T @ S @ Ad)
    S = Qd + Ad.T @ S @ (Ad - Bd @ K) #backward ricatti update
    S = 0.5 * (S + S.T) # forcing symmetry
    K_grid[k] = K
K_grid[-1] = K_grid[-2] #fills final gain with the previous one


def interp_vec(t, arr):
    return np.array([np.interp(t, t_grid, arr[:, j]) for j in range(arr.shape[1])])


def interp_K(t):
    return np.array([[np.interp(t, t_grid, K_grid[:, 0, j]) for j in range(4)]])


def tvlqr_control(t, y4):
    yn = interp_vec(t, y_nom)
    un = np.interp(t, t_grid, u_nom)
    err = y4 - yn
    # Wrap angle error so feedback does not see artificial 2*pi jumps.
    err[2] = np.arctan2(np.sin(err[2]), np.cos(err[2])) # wraps to [-pi, pi]
    u = un - float(interp_K(t) @ err)
    return smooth_limit(u, u_nom_limit)


def eom_tvlqr(t, y4):
    return internal_f(y4, tvlqr_control(t, y4))


# Use the same nominal IC by default. Perturb it to test TVLQR correction.
y0_tvlqr = y0_nom.copy()
sol_tvlqr = solve_ivp(
    eom_tvlqr, (0.0, T_tvlqr), y0_tvlqr,
    method='RK45', rtol=1e-6, atol=1e-8, max_step=0.01, dense_output=True
)

t_tvlqr = np.linspace(0.0, T_tvlqr, 3001)
y_tvlqr = sol_tvlqr.sol(t_tvlqr).T
u_tvlqr = np.array([tvlqr_control(ti, yi) for ti, yi in zip(t_tvlqr, y_tvlqr)])

x_tvlqr = -R * y_tvlqr[:, 0]
x_ref_tvlqr_hist = np.array([x_ref_tvlqr(ti) for ti in t_tvlqr])
err_tvlqr = x_tvlqr - x_ref_tvlqr_hist
mask_tvlqr = t_tvlqr > 5.0
rms_tvlqr = np.sqrt(np.mean(err_tvlqr[mask_tvlqr]**2))
revs_tvlqr = (np.unwrap(y_tvlqr[:, 2])[-1] - np.unwrap(y_tvlqr[:, 2])[0]) / (2.0 * np.pi)
print(f'TVLQR solved | RMS error after 5 s = {rms_tvlqr:.3f} m | pendulum revolutions = {revs_tvlqr:+.1f} | max |u| = {np.max(np.abs(u_tvlqr)):.2f} N.m')


In [ ]:
# Plot TVLQR trajectory tracking results
import numpy as np
import matplotlib.pyplot as plt

x_nom = -R * y_nom[:, 0]
x_tvlqr = -R * y_tvlqr[:, 0]
xd_tvlqr = -R * y_tvlqr[:, 1]
x_ref_plot = np.array([x_ref_tvlqr(ti) for ti in t_tvlqr])
xd_ref_plot = np.array([xd_ref_tvlqr(ti) for ti in t_tvlqr])
revs_plot = (np.unwrap(y_tvlqr[:, 2]) - np.unwrap(y_tvlqr[:, 2])[0]) / (2.0 * np.pi)
err_plot = x_tvlqr - x_ref_plot
mask_plot = t_tvlqr > 5.0
rms_plot = np.sqrt(np.mean(err_plot[mask_plot]**2))

fig, axs = plt.subplots(5, 1, figsize=(9, 11), sharex=True)

axs[0].plot(t_tvlqr, x_tvlqr, label='TVLQR x')
axs[0].plot(t_tvlqr, x_ref_plot, 'r--', label='x reference')
axs[0].set_ylabel('sphere x [m]')
axs[0].legend(fontsize=8); axs[0].grid(True, alpha=0.3)

axs[1].plot(t_tvlqr, xd_tvlqr, label='xdot')
axs[1].plot(t_tvlqr, xd_ref_plot, 'r--', label='xdot reference')
axs[1].set_ylabel('velocity [m/s]')
axs[1].legend(fontsize=8); axs[1].grid(True, alpha=0.3)

axs[2].plot(t_tvlqr, revs_plot)
axs[2].set_ylabel('pendulum revs')
axs[2].grid(True, alpha=0.3)

axs[3].plot(t_tvlqr, y_tvlqr[:, 3], color='teal')
axs[3].set_ylabel(r'$\dot{\theta}_2$ [rad/s]')
axs[3].grid(True, alpha=0.3)

axs[4].plot(t_tvlqr, u_tvlqr, label='TVLQR u')
axs[4].plot(t_grid, u_nom, '--', alpha=0.7, label='nominal u')
axs[4].set_ylabel('internal u [N.m]')
axs[4].set_xlabel(f't [s]   TVLQR RMS error after 5 s = {rms_plot:.3f} m')
axs[4].legend(fontsize=8); axs[4].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()


In [ ]:
# Animate the TVLQR revolving trajectory
animate_hoop_pendulum(t_tvlqr, y_tvlqr[:, :4], R, l, fps=20, save_mp4=False,
                      follow=True, max_frames=800)


## Feedback Linearization - Revolving Pendulum Tracking

Pure input-output feedback linearization of the sphere position is fragile here because the decoupling gain from internal pendulum torque to sphere acceleration becomes very small at some pendulum phases.

This section uses a **regularized feedback-linearization correction** around a revolving nominal motor command. The pendulum still revolves, and the actuator model is still internal:

$$Q = [-u, u]^T$$

The controller computes the local sphere acceleration model

$$\ddot{x} = \alpha( \theta_2,\dot{\theta}_2) + \beta( \theta_2)u$$

then applies a damped inverse correction instead of dividing directly by $\beta$:

$$\Delta u = 
rac{\beta}{\beta^2 + \lambda}\left(v - \ddot{x}_{nom}
\right)$$


In [ ]:
import numpy as np
from scipy.integrate import solve_ivp

M = globals().get("M", 1.0)
m_p = globals().get("m_p", 0.5)
l = globals().get("l", 0.3)
R = globals().get("R", 0.5)
m_h = globals().get("m_h", 0.3)
g = globals().get("g", 9.81)

# Choose: 'smooth_step' or 'sinusoid'
fl_trajectory_type = 'sinusoid'

fl_sine_amplitude = 4.0
fl_sine_frequency = 0.1
fl_sine_offset = 0.0

fl_step_distance = 1.0
fl_step_duration = 8.0


def x_ref_fl(t):
    if fl_trajectory_type == 'sinusoid':
        w = 2.0 * np.pi * fl_sine_frequency
        return fl_sine_offset + fl_sine_amplitude * np.sin(w * t)
    tt = np.minimum(t, fl_step_duration)
    return fl_step_distance * (1.0 - np.cos(np.pi * tt / fl_step_duration)) / 2.0


def xd_ref_fl(t):
    if fl_trajectory_type == 'sinusoid':
        w = 2.0 * np.pi * fl_sine_frequency
        return fl_sine_amplitude * w * np.cos(w * t)
    if t <= fl_step_duration:
        return fl_step_distance * (np.pi / fl_step_duration) * np.sin(np.pi * t / fl_step_duration) / 2.0
    return 0.0


def xdd_ref_fl(t):
    if fl_trajectory_type == 'sinusoid':
        w = 2.0 * np.pi * fl_sine_frequency
        return -fl_sine_amplitude * w**2 * np.sin(w * t)
    if t <= fl_step_duration:
        return fl_step_distance * (np.pi / fl_step_duration)**2 * np.cos(np.pi * t / fl_step_duration) / 2.0
    return 0.0


def smooth_limit(value, limit):
    return limit * np.tanh(value / limit)


def qdd_internal(theta2, theta2dot, u):
    s2 = np.sin(theta2)
    c2 = np.cos(theta2)
    Mred = np.array([
        [(2*m_h + m_p) * R**2,   m_p * l * R * s2],
        [ m_p * l * R * s2,        m_p * l**2      ]
    ])
    Rvec = np.array([
        -u - m_p * l * R * c2 * theta2dot**2,
        u - m_p * g * l * c2
    ])
    return np.linalg.solve(Mred, Rvec)


def xdd_affine_terms(theta2, theta2dot):
    qdd0 = qdd_internal(theta2, theta2dot, 0.0)
    qdd1 = qdd_internal(theta2, theta2dot, 1.0)
    alpha = -R * qdd0[0]
    beta = -R * (qdd1[0] - qdd0[0])
    return alpha, beta


# Revolving nominal command, similar to the PID section.
Kpx_nom_fl = 5.0
Kdx_nom_fl = 3.0
Ktheta_nom_fl = -0.05
Ktheta_dot_nom_fl = 0.01

# Feedback-linearized acceleration-error correction.
Kpx_fl = 5.0
Kdx_fl = 3.0
lambda_fl = 5.0       # larger = safer/damped inverse, smaller = more aggressive
u_max_fl = 20.0


def feedback_linearizing_torque(t, state):
    theta1, theta1dot, theta2, theta2dot, _ = state
    x = -R * theta1
    xdot = -R * theta1dot
    x_ref = x_ref_fl(t)
    xd_ref = xd_ref_fl(t)
    xdd_ref = xdd_ref_fl(t)

    ex = x_ref - x
    exd = xd_ref - xdot

    u_nom = (Kpx_nom_fl * ex + Kdx_nom_fl * exd
             + Ktheta_nom_fl * np.sin(theta2)
             + Ktheta_dot_nom_fl * theta2dot)

    alpha, beta = xdd_affine_terms(theta2, theta2dot)
    xdd_nom = alpha + beta * u_nom
    v = xdd_ref + Kpx_fl * ex + Kdx_fl * exd

    du = beta * (v - xdd_nom) / (beta**2 + lambda_fl)
    u = smooth_limit(u_nom + du, u_max_fl)
    return u, ex, u_nom, du, beta


def eom_feedback_linearized(t, state):
    theta1, theta1dot, theta2, theta2dot, ix = state
    u, ex, _, _, _ = feedback_linearizing_torque(t, state)
    qdd = qdd_internal(theta2, theta2dot, u)
    return [theta1dot, qdd[0], theta2dot, qdd[1], ex]


# Start with spin so the pendulum revolves.
theta2_init_fl = np.pi / 2.0
theta2dot_init_fl = 4.0
y0_fl = [0.0, 0.0, theta2_init_fl, theta2dot_init_fl, 0.0]

sol_fl = solve_ivp(
    eom_feedback_linearized, (0.0, 30.0), y0_fl,
    method='RK45', rtol=1e-5, atol=1e-7, max_step=0.01, dense_output=True
)

t_fl = np.linspace(sol_fl.t[0], sol_fl.t[-1], 3001)
y_fl = sol_fl.sol(t_fl).T

x_fl = -R * y_fl[:, 0]
x_ref_fl_hist = np.array([x_ref_fl(ti) for ti in t_fl])
err_fl = x_fl - x_ref_fl_hist
mask_fl = t_fl > 5.0
rms_fl = np.sqrt(np.mean(err_fl[mask_fl]**2))
revs_fl = (np.unwrap(y_fl[:, 2])[-1] - np.unwrap(y_fl[:, 2])[0]) / (2.0 * np.pi)
print(f'Feedback linearization solved | RMS error after 5 s = {rms_fl:.3f} m | pendulum revolutions = {revs_fl:+.1f}')


In [ ]:
# Plot feedback-linearization trajectory tracking results
import numpy as np
import matplotlib.pyplot as plt

x_fl = -R * y_fl[:, 0]
xd_fl = -R * y_fl[:, 1]
x_ref_fl_hist = np.array([x_ref_fl(ti) for ti in t_fl])
xd_ref_fl_hist = np.array([xd_ref_fl(ti) for ti in t_fl])
revs_fl_hist = (np.unwrap(y_fl[:, 2]) - np.unwrap(y_fl[:, 2])[0]) / (2.0 * np.pi)

u_fl_hist = np.zeros_like(t_fl)
u_nom_fl_hist = np.zeros_like(t_fl)
du_fl_hist = np.zeros_like(t_fl)
beta_fl_hist = np.zeros_like(t_fl)
for i, ti in enumerate(t_fl):
    u_fl_hist[i], _, u_nom_fl_hist[i], du_fl_hist[i], beta_fl_hist[i] = feedback_linearizing_torque(ti, y_fl[i])

err_fl = x_fl - x_ref_fl_hist
mask_fl = t_fl > 5.0
rms_fl = np.sqrt(np.mean(err_fl[mask_fl]**2))

fig, axs = plt.subplots(5, 1, figsize=(9, 11), sharex=True)

axs[0].plot(t_fl, x_fl, label='x')
axs[0].plot(t_fl, x_ref_fl_hist, 'r--', label='x reference')
axs[0].set_ylabel('sphere x [m]')
axs[0].legend(fontsize=8); axs[0].grid(True, alpha=0.3)

axs[1].plot(t_fl, xd_fl, label='xdot')
axs[1].plot(t_fl, xd_ref_fl_hist, 'r--', label='xdot reference')
axs[1].set_ylabel('velocity [m/s]')
axs[1].legend(fontsize=8); axs[1].grid(True, alpha=0.3)

axs[2].plot(t_fl, revs_fl_hist)
axs[2].set_ylabel('pendulum revs')
axs[2].grid(True, alpha=0.3)

axs[3].plot(t_fl, u_fl_hist, label='total u')
axs[3].plot(t_fl, u_nom_fl_hist, '--', alpha=0.7, label='nominal')
axs[3].plot(t_fl, du_fl_hist, ':', alpha=0.8, label='FL correction')
axs[3].set_ylabel('internal u [N.m]')
axs[3].legend(fontsize=8); axs[3].grid(True, alpha=0.3)

axs[4].plot(t_fl, beta_fl_hist)
axs[4].axhline(0.0, color='r', ls='--', alpha=0.4)
axs[4].set_ylabel(r'$\beta$')
axs[4].set_xlabel(f't [s]   FL RMS error after 5 s = {rms_fl:.3f} m')
axs[4].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()


In [ ]:
# Animate the feedback-linearized revolving trajectory
animate_hoop_pendulum(t_fl, y_fl[:, :4], R, l, fps=20, save_mp4=False,
                      follow=True, max_frames=800)


## Computed Torque Controller - Revolving Pendulum Tracking



In [ ]:
import numpy as np
from scipy.integrate import solve_ivp

M = globals().get("M", 1.0)
m_p = globals().get("m_p", 0.5)
l = globals().get("l", 0.3)
R = globals().get("R", 0.5)
m_h = globals().get("m_h", 0.3)
g = globals().get("g", 9.81)

# =============================================================================
# Computed Torque Controller (inverse-dynamics tracking of the sphere)
# Input  : internal motor torque u, Q = [-u, u]^T
# Output : sphere position x  =  -R * theta1
# Pendulum is free to revolve.
#
# The full EOM gives an input-output relation
#     xddot = alpha(state) + beta(state) * u
# so the "computed torque" inverse-dynamics law is
#     u = ( xddot_des - alpha ) / beta
# with xddot_des coming from an outer PD + feedforward on the sphere.
# beta vanishes when sin(theta2) = -l/R, so we use a damped pseudo-inverse.
#
# A small spin-regulating feedforward keeps the pendulum revolving at a
# nominal rate (so beta stays well-conditioned) and stabilizes the
# internal pendulum dynamics, which are unobservable by the outer loop.
# =============================================================================

# Reference trajectory (same options/scale as the FL section)
ctc_trajectory_type = 'sinusoid'   # 'sinusoid' or 'smooth_step'

ctc_sine_amplitude = 4.0
ctc_sine_frequency = 0.1
ctc_sine_offset    = 0.0

ctc_step_distance = 4.0
ctc_step_duration = 8.0


def x_ref_ctc(t):
    if ctc_trajectory_type == 'sinusoid':
        w = 2.0 * np.pi * ctc_sine_frequency
        return ctc_sine_offset + ctc_sine_amplitude * np.sin(w * t)
    tt = np.minimum(t, ctc_step_duration)
    return ctc_step_distance * (1.0 - np.cos(np.pi * tt / ctc_step_duration)) / 2.0


def xd_ref_ctc(t):
    if ctc_trajectory_type == 'sinusoid':
        w = 2.0 * np.pi * ctc_sine_frequency
        return ctc_sine_amplitude * w * np.cos(w * t)
    if t <= ctc_step_duration:
        return ctc_step_distance * (np.pi / ctc_step_duration) * np.sin(np.pi * t / ctc_step_duration) / 2.0
    return 0.0


def xdd_ref_ctc(t):
    if ctc_trajectory_type == 'sinusoid':
        w = 2.0 * np.pi * ctc_sine_frequency
        return -ctc_sine_amplitude * w**2 * np.sin(w * t)
    if t <= ctc_step_duration:
        return ctc_step_distance * (np.pi / ctc_step_duration)**2 * np.cos(np.pi * t / ctc_step_duration) / 2.0
    return 0.0


# Outer PD on sphere tracking error (drives xddot_des)
Kpx_ctc = 5.0
Kdx_ctc = 3.0

# Internal pendulum stabilization (small feedforward, applied through u)
Ktheta_ctc        = -0.05    # on sin(theta2)
Kth2dot_ctc       =  0.01    # on theta2dot

# Inverse-dynamics regularization & saturation
lambda_ctc = 5.0         # Tikhonov damping on 1/beta near singularity sin(th2) = -l/R
u_max_ctc  = 20.0        # internal motor torque saturation [N.m]


def smooth_limit(value, limit):
    return limit * np.tanh(value / limit)


def qdd_internal_ctc(theta2, theta2dot, u):
    s2 = np.sin(theta2); c2 = np.cos(theta2)
    Mred = np.array([
        [(2*m_h + m_p) * R**2,   m_p * l * R * s2],
        [ m_p * l * R * s2,        m_p * l**2      ]
    ])
    Rvec = np.array([
        -u - m_p * l * R * c2 * theta2dot**2,
         u - m_p * g * l * c2
    ])
    return np.linalg.solve(Mred, Rvec)


def xdd_affine_ctc(theta2, theta2dot):
    """xddot = alpha(state) + beta(state) * u."""
    qdd0 = qdd_internal_ctc(theta2, theta2dot, 0.0)
    qdd1 = qdd_internal_ctc(theta2, theta2dot, 1.0)
    alpha = -R * qdd0[0]
    beta  = -R * (qdd1[0] - qdd0[0])
    return alpha, beta


def computed_torque(t, state):
    theta1, theta1dot, theta2, theta2dot, _ = state

    x    = -R * theta1
    xdot = -R * theta1dot

    ex  = x_ref_ctc(t)  - x
    exd = xd_ref_ctc(t) - xdot

    # Nominal torque: outer PD + internal stabilization (keeps pendulum revolving)
    u_nom = (Kpx_ctc * ex
             + Kdx_ctc * exd
             + Ktheta_ctc * np.sin(theta2)
             + Kth2dot_ctc * theta2dot)

    alpha, beta = xdd_affine_ctc(theta2, theta2dot)
    xdd_nom = alpha + beta * u_nom

    # Outer PD + feedforward sphere acceleration command
    v = xdd_ref_ctc(t) + Kpx_ctc * ex + Kdx_ctc * exd

    # Damped inverse-dynamics correction on top of u_nom
    du = beta * (v - xdd_nom) / (beta**2 + lambda_ctc)

    u = smooth_limit(u_nom + du, u_max_ctc)
    return u, ex, v, beta


def eom_computed_torque(t, state):
    theta1, theta1dot, theta2, theta2dot, ix = state
    u, ex, _, _ = computed_torque(t, state)
    qdd = qdd_internal_ctc(theta2, theta2dot, u)
    return [theta1dot, qdd[0], theta2dot, qdd[1], ex]


# Initial condition: start with spin so the pendulum revolves
theta2_init_ctc    = np.pi / 2.0
theta2dot_init_ctc = 4.0
y0_ctc = [0.0, 0.0, theta2_init_ctc, theta2dot_init_ctc, 0.0]

sol_ctc = solve_ivp(
    eom_computed_torque, (0.0, 30.0), y0_ctc,
    method='RK45', rtol=1e-5, atol=1e-7, max_step=0.01, dense_output=True
)

t_ctc = np.linspace(sol_ctc.t[0], sol_ctc.t[-1], 3001)
y_ctc = sol_ctc.sol(t_ctc).T

x_ctc           = -R * y_ctc[:, 0]
x_ref_ctc_hist  = np.array([x_ref_ctc(ti) for ti in t_ctc])
err_ctc         = x_ctc - x_ref_ctc_hist
mask_ctc        = t_ctc > 5.0
rms_ctc         = np.sqrt(np.mean(err_ctc[mask_ctc]**2))
revs_ctc        = (np.unwrap(y_ctc[:, 2])[-1] - np.unwrap(y_ctc[:, 2])[0]) / (2.0 * np.pi)

print(f'Computed-torque solved | RMS error after 5 s = {rms_ctc:.3f} m | pendulum revolutions = {revs_ctc:+.1f}')


In [ ]:
# Plot computed-torque trajectory tracking results
import numpy as np
import matplotlib.pyplot as plt

x_ctc           = -R * y_ctc[:, 0]
xd_ctc          = -R * y_ctc[:, 1]
x_ref_ctc_hist  = np.array([x_ref_ctc(ti)  for ti in t_ctc])
xd_ref_ctc_hist = np.array([xd_ref_ctc(ti) for ti in t_ctc])
revs_ctc_hist   = (np.unwrap(y_ctc[:, 2]) - np.unwrap(y_ctc[:, 2])[0]) / (2.0 * np.pi)

u_ctc_hist     = np.zeros_like(t_ctc)
v_ctc_hist     = np.zeros_like(t_ctc)   # commanded sphere acceleration
alpha_ctc_hist = np.zeros_like(t_ctc)   # commanded pendulum acceleration
beta_ctc_hist  = np.zeros_like(t_ctc)   # denom (B + m_p l^2)
for i, ti in enumerate(t_ctc):
    u_ctc_hist[i], _, v_ctc_hist[i], alpha_ctc_hist[i] = computed_torque(ti, y_ctc[i])
    beta_ctc_hist[i] = m_p * l * R * np.sin(y_ctc[i, 2]) + m_p * l**2

err_ctc  = x_ctc - x_ref_ctc_hist
mask_ctc = t_ctc > 5.0
rms_ctc  = np.sqrt(np.mean(err_ctc[mask_ctc]**2))

fig, axs = plt.subplots(5, 1, figsize=(9, 11), sharex=True)

axs[0].plot(t_ctc, x_ctc, label='x')
axs[0].plot(t_ctc, x_ref_ctc_hist, 'r--', label='x reference')
axs[0].set_ylabel('sphere x [m]')
axs[0].legend(fontsize=8); axs[0].grid(True, alpha=0.3)

axs[1].plot(t_ctc, xd_ctc, label='xdot')
axs[1].plot(t_ctc, xd_ref_ctc_hist, 'r--', label='xdot reference')
axs[1].set_ylabel('velocity [m/s]')
axs[1].legend(fontsize=8); axs[1].grid(True, alpha=0.3)

axs[2].plot(t_ctc, revs_ctc_hist)
axs[2].set_ylabel('pendulum revs')
axs[2].grid(True, alpha=0.3)

axs[3].plot(t_ctc, u_ctc_hist, label='total u')
axs[3].plot(t_ctc, v_ctc_hist, '--', alpha=0.7, label=r'$\ddot{x}_{des}$')
axs[3].plot(t_ctc, alpha_ctc_hist, ':', alpha=0.8, label=r'$\ddot{\theta}_{2,des}$')
axs[3].axhline( u_max_ctc, color='r', ls='--', alpha=0.5)
axs[3].axhline(-u_max_ctc, color='r', ls='--', alpha=0.5)
axs[3].set_ylabel('internal u [N.m]')
axs[3].legend(fontsize=8); axs[3].grid(True, alpha=0.3)

axs[4].plot(t_ctc, beta_ctc_hist)
axs[4].axhline(0.0, color='r', ls='--', alpha=0.4)
axs[4].set_ylabel(r'$B + m_p l^2$')
axs[4].set_xlabel(f't [s]   CT RMS error after 5 s = {rms_ctc:.3f} m')
axs[4].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()


In [ ]:
# Animate the computed-torque revolving trajectory
animate_hoop_pendulum(t_ctc, y_ctc[:, :4], R, l, fps=60, save_mp4=False,
                      follow=True, max_frames=800)


## MPC Controller

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
from scipy.linalg import expm, block_diag
from scipy.optimize import minimize

M = globals().get("M", 1.0)
m_p = globals().get("m_p", 0.5)
l = globals().get("l", 0.3)
R = globals().get("R", 0.5)
m_h = globals().get("m_h", 0.3)
g = globals().get("g", 9.81)


# Reference trajectory (matches the other controller sections)
mpc_trajectory_type = 'sinusoid'   # 'sinusoid' or 'smooth_step'

mpc_sine_amplitude = 4.0
mpc_sine_frequency = 0.1
mpc_sine_offset    = 0.0

mpc_step_distance = 4.0
mpc_step_duration = 8.0


def x_ref_mpc(t):
    if mpc_trajectory_type == 'sinusoid':
        w = 2.0 * np.pi * mpc_sine_frequency
        return mpc_sine_offset + mpc_sine_amplitude * np.sin(w * t)
    tt = np.minimum(t, mpc_step_duration)
    return mpc_step_distance * (1.0 - np.cos(np.pi * tt / mpc_step_duration)) / 2.0


def xd_ref_mpc(t):
    if mpc_trajectory_type == 'sinusoid':
        w = 2.0 * np.pi * mpc_sine_frequency
        return mpc_sine_amplitude * w * np.cos(w * t)
    if t <= mpc_step_duration:
        return mpc_step_distance * (np.pi / mpc_step_duration) * np.sin(np.pi * t / mpc_step_duration) / 2.0
    return 0.0


# MPC tuning
dt_mpc       = 0.05              # control / discretization period [s]
T_mpc        = 30.0              # total sim time [s]
N_mpc        = 22                # prediction horizon (steps)
u_max_mpc    = 20.0              # torque saturation [N.m]

Q_y_mpc  = np.diag([5.0, 3.0])
Qf_y_mpc = 5.0 * Q_y_mpc
R_u_mpc  = 1.0
R_du_mpc = 4.0

# Tiny phase feedforward only -- spin cap removed (it fought MPC).
Ktheta_mpc     = -0.05
Ktheta_dot_mpc =  0.0


def smooth_limit(value, limit):
    return limit * np.tanh(value / limit)


M_eff_mpc = (2.0 * m_h + m_p) * R

A_cart = np.array([[0.0, 1.0],
                   [0.0, 0.0]])
B_cart = np.array([[0.0],
                   [1.0 / M_eff_mpc]])

def _zoh(A, B, dt):
    nx = A.shape[0]; nu = B.shape[1]
    aug = np.zeros((nx + nu, nx + nu))
    aug[:nx, :nx]      = A
    aug[:nx, nx:nx+nu] = B
    E = expm(aug * dt)
    return E[:nx, :nx], E[:nx, nx:nx+nu]

Ad_cart, Bd_cart = _zoh(A_cart, B_cart, dt_mpc)

C_cart = np.eye(2)
ny = 2
N   = N_mpc

Sx = np.zeros((N * 2, 2))
Su = np.zeros((N * 2, N))
Apow = np.eye(2)
for k in range(N):
    Apow = Ad_cart @ Apow
    Sx[k*2:(k+1)*2, :] = Apow
    for i in range(k + 1):
        Su[k*2:(k+1)*2, i:i+1] = np.linalg.matrix_power(Ad_cart, k - i) @ Bd_cart

Cbig    = block_diag(*([C_cart] * N))
Y_mat   = Cbig @ Su
Qbig0   = block_diag(*([Q_y_mpc] * (N - 1) + [Qf_y_mpc]))
Ru      = R_u_mpc * np.eye(N)
D_du    = np.eye(N)
D_du[1:, :-1] -= np.eye(N - 1)
bounds_mpc    = [(-u_max_mpc, u_max_mpc)] * N


def mpc_step(z_cur, u_prev, u_warm, ref_y_seq):
    Y_const = Cbig @ (Sx @ z_cur)
    e0 = np.zeros(N); e0[0] = u_prev

    def cost(u):
        Y    = Y_mat @ u + Y_const
        eY   = Y - ref_y_seq
        du   = D_du @ u - e0
        return float(eY @ Qbig0 @ eY + u @ Ru @ u + R_du_mpc * du @ du)

    def jac(u):
        Y    = Y_mat @ u + Y_const
        eY   = Y - ref_y_seq
        du   = D_du @ u - e0
        return 2.0 * (Y_mat.T @ (Qbig0 @ eY) + Ru @ u + R_du_mpc * (D_du.T @ du))

    result = minimize(cost, u_warm, jac=jac, bounds=bounds_mpc, method='SLSQP',
                      options={'maxiter': 60, 'ftol': 1e-6, 'disp': False})
    if result.success:
        u_seq = result.x
    else:
        u_seq = np.clip(u_warm, -u_max_mpc, u_max_mpc)
    return u_seq


# Bearing friction at the pendulum pivot. Physical: real bearings have
# viscous losses that prevent the pendulum from spinning up indefinitely
# (without this the MPC's internal torque drives both the sphere AND the
# pendulum spin, since u is the internal reaction torque between them).
b_pend_mpc = 0.05   # [N.m.s/rad]  light viscous damping on pendulum

def f_dyn_mpc(state, u):
    th1, th1dot, th2, th2dot = state
    s2 = np.sin(th2); c2 = np.cos(th2)
    Mred = np.array([
        [(2*m_h + m_p) * R**2,   m_p * l * R * s2],
        [ m_p * l * R * s2,        m_p * l**2      ]
    ])
    Rvec = np.array([
        -u - m_p * l * R * c2 * th2dot**2,
         u - m_p * g * l * c2 - b_pend_mpc * th2dot
    ])
    qdd = np.linalg.solve(Mred, Rvec)
    return np.array([th1dot, qdd[0], th2dot, qdd[1]])


def eom_mpc(t, state, u):
    return f_dyn_mpc(np.asarray(state[:4]), u)


# Lowered initial spin from 4.0 -> 2.0 rad/s for a more realistic launch speed.
theta2_init_mpc    = np.pi / 2.0
theta2dot_init_mpc = 2.0
y0_native = np.array([0.0, 0.0, theta2_init_mpc, theta2dot_init_mpc])

sim_steps = int(round(T_mpc / dt_mpc))
t_hist = np.zeros(sim_steps + 1)
y_hist = np.zeros((sim_steps + 1, 4))
u_hist = np.zeros(sim_steps)
x_ref_mpc_hist  = np.zeros(sim_steps + 1)
xd_ref_mpc_hist = np.zeros(sim_steps + 1)

y_cur  = y0_native.copy()
u_prev = 0.0
u_warm = np.zeros(N_mpc)

y_hist[0] = y_cur
x_ref_mpc_hist[0]  = x_ref_mpc(0.0)
xd_ref_mpc_hist[0] = xd_ref_mpc(0.0)

import time as _time
t_start = _time.time()
for k in range(sim_steps):
    t_k = k * dt_mpc

    z_cur = np.array([-R * y_cur[0], -R * y_cur[1]])

    ref_y_seq = np.zeros(N * ny)
    for j in range(N):
        tj = t_k + (j + 1) * dt_mpc
        ref_y_seq[ny*j + 0] = x_ref_mpc(tj)
        ref_y_seq[ny*j + 1] = xd_ref_mpc(tj)

    u_seq = mpc_step(z_cur, u_prev, u_warm, ref_y_seq)
    u_warm = np.concatenate((u_seq[1:], [u_seq[-1]]))

    u_ff = (Ktheta_mpc * np.sin(y_cur[2])
            + Ktheta_dot_mpc * y_cur[3])
    u_apply = float(smooth_limit(u_seq[0] + u_ff, u_max_mpc))
    u_hist[k] = u_apply

    sol = solve_ivp(lambda tt, ss: eom_mpc(tt, ss, u_apply),
                    (t_k, t_k + dt_mpc), y_cur,
                    method='RK45', rtol=1e-6, atol=1e-8, max_step=dt_mpc)
    y_cur  = sol.y[:, -1]
    u_prev = u_apply

    t_hist[k + 1] = t_k + dt_mpc
    y_hist[k + 1] = y_cur
    x_ref_mpc_hist[k + 1]  = x_ref_mpc(t_k + dt_mpc)
    xd_ref_mpc_hist[k + 1] = xd_ref_mpc(t_k + dt_mpc)

wall_s = _time.time() - t_start

t_mpc = t_hist
y_mpc = y_hist
x_mpc  = -R * y_mpc[:, 0]
xd_mpc = -R * y_mpc[:, 1]
err_mpc  = x_mpc - x_ref_mpc_hist
mask_mpc = t_mpc > 5.0
rms_mpc  = np.sqrt(np.mean(err_mpc[mask_mpc]**2))
revs_mpc = (np.unwrap(y_mpc[:, 2])[-1] - np.unwrap(y_mpc[:, 2])[0]) / (2.0 * np.pi)
peak_spin_mpc = float(np.max(np.abs(y_mpc[:, 3])))

print(f'MPC solved | RMS error after 5 s = {rms_mpc:.3f} m | pendulum revolutions = {revs_mpc:+.1f}')
print(f'peak |theta2dot| = {peak_spin_mpc:.2f} rad/s')
print(f'horizon N = {N_mpc} steps, dt = {int(dt_mpc*1000)} ms, wall time = {wall_s:.1f} s')


In [ ]:
# Plot MPC trajectory tracking results
import numpy as np
import matplotlib.pyplot as plt

x_pos_mpc  = -R * y_mpc[:, 0]
xd_pos_mpc = -R * y_mpc[:, 1]
revs_trace = (np.unwrap(y_mpc[:, 2]) - np.unwrap(y_mpc[:, 2])[0]) / (2.0 * np.pi)

# pad u history so it lines up with t_mpc (control held over each interval)
u_mpc_plot = np.concatenate(([u_hist[0]], u_hist))

fig, axs = plt.subplots(4, 1, figsize=(9, 9), sharex=True)

axs[0].plot(t_mpc, x_pos_mpc, label='x')
axs[0].plot(t_mpc, x_ref_mpc_hist, 'r--', label='x reference')
axs[0].set_ylabel('sphere x [m]')
axs[0].legend(fontsize=8); axs[0].grid(True, alpha=0.3)

axs[1].plot(t_mpc, xd_pos_mpc, label='xdot')
axs[1].plot(t_mpc, xd_ref_mpc_hist, 'r--', label='xdot reference')
axs[1].set_ylabel('velocity [m/s]')
axs[1].legend(fontsize=8); axs[1].grid(True, alpha=0.3)

axs[2].plot(t_mpc, revs_trace)
axs[2].set_ylabel('pendulum revs')
axs[2].grid(True, alpha=0.3)

axs[3].step(t_mpc, u_mpc_plot, where='post', label='MPC u')
axs[3].axhline( u_max_mpc, color='r', ls='--', alpha=0.5)
axs[3].axhline(-u_max_mpc, color='r', ls='--', alpha=0.5)
axs[3].set_ylabel('internal u [N.m]')
axs[3].set_xlabel(f't [s]   MPC RMS error after 5 s = {rms_mpc:.3f} m')
axs[3].legend(fontsize=8); axs[3].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()


In [ ]:
# Animate the MPC revolving trajectory
animate_hoop_pendulum(t_mpc, y_mpc, R, l, fps=60, save_mp4=False,
                      follow=True, max_frames=800)


## Controller Comparison

In [ ]:
# ---------------------------------------------------------------
# Controller comparison: overlay sphere x(t) for every revolving
# controller against the tracking reference.
# ---------------------------------------------------------------
import matplotlib.pyplot as plt
import numpy as np

controllers = [
    ("PID (revolving)", t_track, x_track, x_ref_hist,       "#1f77b4"),
    ("TVLQR",           t_tvlqr, x_tvlqr, x_ref_tvlqr_hist, "#2ca02c"),
    ("Feedback Lin.",   t_fl,    x_fl,    x_ref_fl_hist,    "#ff7f0e"),
    ("Computed Torque", t_ctc,   x_ctc,   x_ref_ctc_hist,   "#9467bd"),
    ("MPC",             t_mpc,   x_mpc,   x_ref_mpc_hist,   "#d62728"),
]

# Use the longest trace as the canonical reference.
ref_name, ref_t, _, ref_x, _ = max(controllers, key=lambda c: c[1][-1])

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(ref_t, ref_x, "k--", lw=2.0, label=f"Reference ({ref_name})", zorder=10)

for name, t_c, x_c, _, color in controllers:
    rms = float(np.sqrt(np.mean((x_c - np.interp(t_c, ref_t, ref_x)) ** 2)))
    ax.plot(t_c, x_c, color=color, lw=1.4, label=f"{name}  (RMS={rms:.3f} m)")

ax.set_xlabel("time [s]")
ax.set_ylabel("sphere position  x  [m]")
ax.set_title("Controller comparison: sphere position vs reference")
ax.grid(True, alpha=0.3)
ax.legend(loc="best", fontsize=9)
plt.tight_layout()
plt.show()
